<a href="https://colab.research.google.com/github/Samarjamal326/Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

**Lane:** CTR / Engagement Opportunity Scoring

This notebook turns the ML-03 framing into a reproducible data contract. The goal is to rank content items by **future search CTR opportunity** using only information that would have been available at the decision cutoff.

The warehouse is used for the full-release contract; the bundled starter CSV is not used for the warehouse verification cells below.

## 1. Unit of analysis + time window

**One row = one client × content item.**

The source fact table is daily-grain (`client × content × report_date`), so the contract aggregates daily observations into two non-overlapping monthly windows:

| Window | Dates | Role |
|---|---|---|
| February 2026 | 2026-02-01 → 2026-02-28 | **Features / decision information** |
| March 2026 | 2026-03-01 → 2026-03-31 | **Future label** |

The decision cutoff is **2026-02-28**. A feature must be knowable by that date. The March outcome is deliberately kept separate so that the target is observed after the decision moment.

**Target / ranking outcome:** `future_ctr = March clicks / March impressions`, for content with measured March data and at least 100 March impressions. The ranking objective is to prioritize pages whose observed future CTR indicates greater opportunity relative to peers; model performance will be evaluated with a ranking metric in later work.

**Deliberate exclusion:** pseudonymous `client_hash_id` and `content_hash_id` are retained only for grouping/joining/validation, never as model features. Raw queries, URLs, client names, and other private identifiers are also outside this public contract.

## 2. Fields: feature / label / context / excluded

### Feature candidates — maximum five

1. `feb_impressions` — February measured search impressions.
2. `feb_clicks` — February measured search clicks.
3. `feb_ctr` — February clicks / impressions.
4. `feb_avg_position` — February impression-weighted average position.
5. `content_age_days_feb` — content age at the February 28 cutoff.

Each is available before the label window:

- **`feb_impressions`:** known from measured February Search Console observations by the cutoff.
- **`feb_clicks`:** known from measured February Search Console observations by the cutoff.
- **`feb_ctr`:** computed only from February clicks and impressions, so it uses no March information.
- **`feb_avg_position`:** computed only from February position observations and February impressions.
- **`content_age_days_feb`:** computed from the content creation date and the February 28 cutoff.

### Label

- `future_ctr` = March measured clicks / March measured impressions, with a minimum of 100 March impressions to avoid an extremely noisy denominator.

### Context / validation only

- `client_hash_id`
- `content_hash_id`
- `report_date`
- `gsc_data_available`
- measured-day counts

These help define, audit, join, and validate the dataset. They are not model features.

### Excluded

- `trend_direction`, `trend_pct`, or any field calculated from the same/future comparison window as the label.
- Product decision flags / composite scores.
- Pseudonymous IDs as model inputs.
- Raw private queries, URLs, domains, or client identifiers.

The key rule is **cutoff discipline**: anything that depends on March, or is itself a downstream label/decision, cannot be a February feature.

## 3. Setup

Run this cell first. In Colab, store the approved Hugging Face **READ** token as a Secret named `HF_TOKEN`. The token is passed through a DuckDB session variable rather than written into the SQL source.

In [2]:
# Colab setup — no token is hard-coded.
!pip -q install duckdb pandas scipy

import os
import duckdb
import pandas as pd
import numpy as np

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. In Colab: left sidebar → Secrets → add HF_TOKEN "
        "using your approved Hugging Face READ token."
    )

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Keep the credential out of the notebook source and out of query strings.
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])
con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN getvariable('hf_token'))"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"
DIM = f"read_parquet('{REL}/dim_content.parquet')"

print("Connected.")
print("Feature window: February 2026")
print("Label window: March 2026")


Connected.
Feature window: February 2026
Label window: March 2026


### Verification query 1 — grain

This checks that the daily fact slice has one observation per `client_hash_id × content_hash_id × report_date` by comparing total rows with distinct three-column keys.

In [3]:
grain_check = con.sql(f"""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || CAST(report_date AS VARCHAR))
        AS distinct_daily_keys,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id) AS distinct_content_client_pairs,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {FEB}
""").df()

display(grain_check)
print(
    "Grain check:",
    "PASS — one row per client × content × day"
    if grain_check.loc[0, "rows"] == grain_check.loc[0, "distinct_daily_keys"]
    else "INVESTIGATE — duplicate daily keys detected"
)


,rows,distinct_daily_keys,distinct_content_client_pairs,min_date,max_date
0,7355108,7355108,321546,2026-02-01,2026-02-28


Grain check: PASS — one row per client × content × day


### Verification query 2 — February slice count and date span

This confirms that the feature partition is the intended February 2026 slice and records its observed row count/date span.

In [4]:
feb_slice = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date,
    COUNT(DISTINCT report_date) AS distinct_dates
FROM {FEB}
""").df()

display(feb_slice)


,row_count,min_report_date,max_report_date,distinct_dates
0,7355108,2026-02-01,2026-02-28,28


### Verification query 3 — measured-data availability

`gsc_data_available` must be checked with **`IS TRUE`**. A false value means the metric was not measured; it must not be silently interpreted as zero traffic.

In [5]:
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS measured_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS not_measured_rows,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) / COUNT(*),
        2
    ) AS measured_pct
FROM {FEB}
""").df()

display(availability_check)


,total_rows,measured_rows,not_measured_rows,measured_pct
0,7355108,2621783,4733325,35.65


## 4. Five-feature frame

The following feature query uses only February observations and the content creation date. It explicitly keeps measured data separate from unavailable data.

For position, the contract uses an **impression-weighted** average rather than an unweighted mean across daily rows.

In [6]:
feature_frame = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS feb_impressions,
        SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) AS feb_clicks,
        SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE) AS feb_sum_position,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS feb_measured_days
    FROM {FEB}
    GROUP BY 1, 2
),
content AS (
    SELECT
        content_hash_id,
        content_created_date
    FROM {DIM}
)
SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.feb_impressions,
    f.feb_clicks,
    f.feb_clicks / NULLIF(f.feb_impressions, 0) AS feb_ctr,
    f.feb_sum_position / NULLIF(f.feb_impressions, 0) AS feb_avg_position,
    DATE_DIFF(
        'day',
        CAST(c.content_created_date AS DATE),
        DATE '2026-02-28'
    ) AS content_age_days_feb,
    f.feb_measured_days
FROM feb f
LEFT JOIN content c USING (content_hash_id)
WHERE f.feb_measured_days > 0
  AND f.feb_impressions > 0
""").df()

print(f"Feature rows: {len(feature_frame):,}")
display(feature_frame.head())


Feature rows: 153,559


,client_hash_id,content_hash_id,feb_impressions,feb_clicks,feb_ctr,feb_avg_position,content_age_days_feb,feb_measured_days
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,0.000000,12.448161,226,28
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,0.008186,6.316508,226,28
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,0.000000,9.966926,226,28
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,0.001024,41.814739,226,28
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,0.002062,10.307216,226,28


## 5. Label frame — future CTR

The label is calculated from the **March** window only. We preserve the availability flag, count measured days, and exclude content with zero measured March days rather than turning "not measured" into zero traffic.

A 100-impression minimum is applied to the observed March outcome so that the future CTR is not dominated by tiny denominators.

In [7]:
label_frame = con.sql(f"""
WITH mar AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS mar_impressions,
        SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) AS mar_clicks,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS mar_measured_days
    FROM {MAR}
    GROUP BY 1, 2
)
SELECT
    client_hash_id,
    content_hash_id,
    mar_impressions,
    mar_clicks,
    mar_measured_days,
    mar_clicks / NULLIF(mar_impressions, 0) AS future_ctr
FROM mar
WHERE mar_measured_days > 0
  AND mar_impressions >= 100
""").df()

print(f"Label rows with measured March data and ≥100 impressions: {len(label_frame):,}")
display(label_frame.head())


Label rows with measured March data and ≥100 impressions: 101,441


,client_hash_id,content_hash_id,mar_impressions,mar_clicks,mar_measured_days,future_ctr
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,31,0.001073
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,31,0.000000
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,31,0.001066
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,31,0.002629
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,31,0.002331


## 6. Contract merge and final five-feature check

In [8]:
contract_frame = feature_frame.merge(
    label_frame,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

FEATURES = [
    "feb_impressions",
    "feb_clicks",
    "feb_ctr",
    "feb_avg_position",
    "content_age_days_feb",
]
TARGET = "future_ctr"

assert len(FEATURES) == 5
assert not set(FEATURES) & {TARGET, "mar_impressions", "mar_clicks", "mar_measured_days"}
assert contract_frame[FEATURES].notna().all().all(), "Unexpected feature missingness"

print(f"Final contract rows: {len(contract_frame):,}")
print("Features:", FEATURES)
print("Target:", TARGET)
display(contract_frame[["client_hash_id", "content_hash_id"] + FEATURES + [TARGET]].head())


Final contract rows: 86,560
Features: ['feb_impressions', 'feb_clicks', 'feb_ctr', 'feb_avg_position', 'content_age_days_feb']
Target: future_ctr


,client_hash_id,content_hash_id,feb_impressions,feb_clicks,feb_ctr,feb_avg_position,content_age_days_feb,future_ctr
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,0.000000,12.448161,226,0.000000
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,0.008186,6.316508,226,0.000275
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,0.000000,9.966926,226,0.000000
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,0.001024,41.814739,226,0.001065
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,0.002062,10.307216,226,0.000996


## 7. Deliberate leakage experiment

This is an intentional failure test.

We compare an honest February feature (`feb_ctr`) with a deliberately leaked feature that is **exactly the March target** (`future_ctr`) while predicting `future_ctr`.

The leaked feature should have a perfect rank relationship with the target. That is the warning sign: it uses information that was not available at the February 28 decision cutoff.

The leaked column is **not** retained in the contract feature list.

In [9]:
from scipy.stats import spearmanr

honest_rho, honest_p = spearmanr(
    contract_frame["feb_ctr"],
    contract_frame[TARGET],
    nan_policy="omit",
)

# Deliberate leak: this is the future label itself pretending to be a feature.
contract_frame["LEAK_future_ctr"] = contract_frame[TARGET]

leak_rho, leak_p = spearmanr(
    contract_frame["LEAK_future_ctr"],
    contract_frame[TARGET],
    nan_policy="omit",
)

print(f"Honest February CTR vs future CTR Spearman rho: {honest_rho:.3f}")
print(f"Leaked future CTR vs future CTR Spearman rho:  {leak_rho:.3f}")

assert abs(leak_rho - 1.0) < 1e-12, "Leak experiment did not behave as expected."

# Delete the leaked feature: it is not part of the data contract.
contract_frame = contract_frame.drop(columns=["LEAK_future_ctr"])

print("Leak experiment complete: leaked feature deleted.")
print("Final features:", FEATURES)


Honest February CTR vs future CTR Spearman rho: 0.503
Leaked future CTR vs future CTR Spearman rho:  1.000
Leak experiment complete: leaked feature deleted.
Final features: ['feb_impressions', 'feb_clicks', 'feb_ctr', 'feb_avg_position', 'content_age_days_feb']


## 8. Data limits

- This contract observes search/engagement measurements; it does **not** establish causality.
- `gsc_data_available = FALSE` means the metric was not measured, not that impressions or clicks were zero.
- The February feature window and March label window are deliberately separated; later model validation must preserve that temporal discipline.
- The 100-impression label threshold improves stability but changes the population being modeled: very-low-volume content is not part of this label definition.
- The public repository must not contain client names, domains, URLs, raw queries, or other private identifiers.
- A successful model would support **prioritization / decision support**, not a claim that it predicts Google's ranking algorithm or proves that a content change caused a CTR change.

## Self-check

- [x] One row is defined as client × content.
- [x] February is the feature window; March is the future label window.
- [x] Three small verification queries are shown with outputs.
- [x] Availability uses `gsc_data_available IS TRUE`.
- [x] Five features are defined, with a cutoff justification for each.
- [x] Pseudonymous IDs are grouping/joining keys, not model features.
- [x] A deliberate future-label leakage experiment is shown and the leaked feature is deleted.
- [x] The notebook uses careful, decision-support language.
- [ ] Run all cells in Colab and commit this notebook to `work/notebooks/w03_data_contract.ipynb` in the Flyrank repo.
